# 02 — Data Preparation

Extracción masiva de todas las rutas ópticas y su historial de mediciones,
seguida del pipeline completo de feature engineering (`src/features.py`) y de
las weak labels (`src/weak_labels.py`).

**Antes de correr esto sobre todas las rutas**, se prueba primero sobre un
subconjunto pequeño para medir tiempo/volumen (ver sección 2).


In [ ]:
import sys
import logging
from pathlib import Path

SRC = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

from fms_client import FmsClient
import fms_extract as FE
import features as F
import weak_labels as WL

client = FmsClient()

RAW_ROUTES_PATH   = Path("../data/raw/optical_routes.parquet")
RAW_RESULTS_PATH  = Path("../data/raw/results_all.parquet")
SAMPLE_RESULTS_PATH = Path("../data/raw/results_sample.parquet")
STATE_PATH        = Path("../data/interim/extraction_state.json")
STATE_SAMPLE_PATH = Path("../data/interim/extraction_state_sample.json")
PROCESSED_DIR     = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Extraer todas las rutas ópticas

In [ ]:
df_routes = FE.extract_all_routes(client, out_path=RAW_ROUTES_PATH)
print(f"{len(df_routes)} rutas ópticas encontradas.")
df_routes.head()

## 2. Prueba de extracción a pequeña escala

Corré esto primero sobre un subconjunto de rutas para verificar que la
conexión al FMS funciona y estimar cuánto tardará la extracción completa.

Si el parquet de muestra no existe, el estado se resetea automáticamente
y la descarga arranca de cero — no hace falta borrar nada a mano.

In [ ]:
import time

sample_ids = df_routes["id"].head(4).tolist()
print(f"Extrayendo resultados de {len(sample_ids)} rutas de ejemplo: {sample_ids}")
t0 = time.time()
df_sample = FE.extract_results_for_routes(
    client, sample_ids,
    out_path=SAMPLE_RESULTS_PATH,
    state_path=STATE_SAMPLE_PATH,
)
elapsed = time.time() - t0
if df_sample.empty or "route_id" not in df_sample.columns:
    print("ADVERTENCIA: No se extrajeron mediciones en la prueba. Revisá el log de arriba.")
else:
    print(f"{len(df_sample)} mediciones extraídas de {df_sample['route_id'].nunique()} rutas "
          f"en {elapsed:.1f}s ({elapsed/max(len(sample_ids),1):.1f}s/ruta)")

## 3. Extracción completa (todas las rutas)

Extrae el historial completo de las 4 rutas y lo guarda en
`data/raw/results_all.parquet`.

**Resumible:** si se interrumpe (Ctrl+C / kernel restart), volver a correr
esta celda continúa desde donde quedó. Si `results_all.parquet` no existe,
el estado se resetea solo y arranca de cero.

In [ ]:
all_route_ids = df_routes["id"].tolist()

df_results = FE.extract_results_for_routes(
    client, all_route_ids,
    out_path=RAW_RESULTS_PATH,
    state_path=STATE_PATH,
    sleep_between_calls=0.3,
)

if df_results.empty or "route_id" not in df_results.columns:
    raise RuntimeError(
        "No se extrajeron mediciones. Revisá el log de arriba para ver qué ruta falló "
        "y por qué. Si el error fue de red o de token, volvé a correr esta celda."
    )

print(f"{len(df_results)} mediciones extraídas en total, de {df_results['route_id'].nunique()} rutas.")
df_results.head()

## 4. Resolver baselines no encontrados dentro del propio historial extraído

In [ ]:
df_extra_baselines = FE.resolve_unmatched_baselines(client, df_results)
print(f"{len(df_extra_baselines)} baselines adicionales resueltos puntualmente.")

if not df_extra_baselines.empty:
    df_results = (
        pd.concat([df_results, df_extra_baselines], ignore_index=True)
        .drop_duplicates(subset="resultid", keep="last")
    )
    df_results.to_parquet(RAW_RESULTS_PATH, index=False)


## 5. Feature engineering

In [ ]:
df_meas_features = F.build_measurement_features(df_results)
df_baseline_lookup = F.build_baseline_lookup(df_meas_features)
df_delta = F.build_delta_features(df_meas_features, df_baseline_lookup)
df_timeline = F.build_route_timeline_features(df_delta, window=5)

print(f"baseline_resolved=True en {df_timeline['baseline_resolved'].mean():.1%} de las mediciones")
df_timeline.head()


## 6. Weak labels

In [ ]:
df_timeline["at_risk"] = WL.label_at_risk(df_timeline)
df_timeline["severity"] = WL.label_severity(df_timeline)

print(df_timeline["at_risk"].value_counts(normalize=True))
print(df_timeline["severity"].value_counts())


## 7. Guardar datasets procesados

In [ ]:
df_meas_features.to_parquet(PROCESSED_DIR / "features_by_measurement.parquet", index=False)
df_timeline.to_parquet(PROCESSED_DIR / "features_by_route_timeline.parquet", index=False)
print("[+] Guardado en data/processed/")


## 8. Reporte de calidad de datos

In [ ]:
print("Rutas con baseline encontrado (BaselineId resuelto a un resultid):")
display(df_timeline.groupby("route_id")["baseline_found"].mean())

print("\nRutas con baseline ademas utilizable (con total_loss_db numerico):")
display(df_timeline.groupby("route_id")["baseline_resolved"].mean())

print("\nTasa de distance_capped por ruta (medicion fuera del alcance confiable")
print("del OTDR, ver notebook 01 - seccion 2 para la causa raiz; se uso")
print("elements_loss_sum_db como respaldo de total_loss_db en estas filas):")
display(df_timeline.groupby("route_id")["distance_capped"].mean())

print("\nOutliers de Loss (>50dB o negativos):")
display(df_timeline.loc[(df_timeline["total_loss_db"] > 50) | (df_timeline["total_loss_db"] < 0)])

print("\nBalance de clases (at_risk):")
display(df_timeline["at_risk"].value_counts())
